Handling of images once they have been downloaded with planet_download.ipynb up to a product allowing us to train a model. 

Replaces the run_script_AF.py once created but not needed to re-run everything. 

Will apply the extraction from the zip file of downloaded images from Planet; Prepare the images creating the PNG images from the tif file allowing for the manual annotation; and will segment the images describe them for check, and cull them to suit the model requirements, and finally apply the model training. 

In [ ]:
import sys
import os
import yaml

# Add the project root to sys.path (adjust as needed)
sys.path.append(os.path.abspath("counting_waterholes"))


import counting_boats.boat_utils.planet_utils as planet_utils
import counting_boats.boat_utils.testing as testing

Extraction of the file composite.tif obtained from the planet order and downloaded into the zip file. Renders a tif file and renames it with the date_aoi.tif outside the zip file. 

In [ ]:
import os
import yaml

import counting_boats.boat_utils.planet_utils

# Define the path to your zip file
zip_path = "images/raw_images"


# Run extraction
counting_boats.boat_utils.planet_utils.extract_zip(zip_path)

Prepare the raw tif image into a usable png in future steps. Creates a padded png image to exactly match the size dividable by the stride and tile size. 
Need to define the config to make sure it matches my paths and running the tif to png transformation. 

In [ ]:
import os
import yaml

import counting_boats.train

# #cfg config:
# with open("config_train_GPU.yaml", "r") as ymlfile:
#     cfg = yaml.load(ymlfile, Loader=yaml.FullLoader)
#     os.makedirs(cfg["output_dir"], exist_ok=True)
#     cfg["tif_dir"] = cfg.get(
#         "tif_dir", os.path.join(cfg["proj_root"], "images", "RawImages")
#     )  # This is generated so not included in the config file



#Run preparation of the tif files into png and renamed the tif. 
#prepare(r"C:\Users\adria\OneDrive - AdrianoFossati\Documents\MASTER Australia\RA\Waterholes_project\counting_waterholes\images\RawImages", cfg)
#Use relative paths not absolute 
counting_boats.train.prepare_S2("config_train_Drive_SF.yaml")
 

### To check the scale of values in the Sentinel-2 images 

For adjusting the scaling in the image_cutting_support.create_padded_png_S2 function

In [ ]:
# def inspect_tif_values(tif_path):
#     """Print min, max, and percentile values for each band in a TIFF file using rasterio"""
#     import numpy as np
#     import rasterio
    
#     with rasterio.open(tif_path) as src:
#         print(f"Inspecting: {tif_path}")
        
#         for band_idx in range(1, src.count + 1):
#             band_data = src.read(band_idx)
#             # Exclude NoData values
#             if src.nodata is not None:
#                 band_data = band_data[band_data != src.nodata]
#             else:
#                 # If no NoData value is set, exclude zeros as a common default
#                 band_data = band_data[band_data > 0]
            
#             if band_data.size > 0:
#                 min_val = np.min(band_data)
#                 max_val = np.max(band_data)
#                 p95 = np.percentile(band_data, 95)  # 95th percentile
#                 p99 = np.percentile(band_data, 99)  # 99th percentile
#                 mean_val = np.mean(band_data)
                
#                 print(f"Band {band_idx}: Min={min_val:.4f}, Max={max_val:.4f}")
#                 print(f"         Mean={mean_val:.4f}, 95th={p95:.4f}, 99th={p99:.4f}")
#             else:
#                 print(f"Band {band_idx}: No valid data")


# # Define the path to your TIFF file
# tif_path = "images/planet_2_sentinel/tifs/mimal_test_2024-06.tif"
# inspect_tif_values(tif_path)

Once the png is created, as we are in the training of the model phase, I need to go on LabelMe (called here in the terminal) and manually annotate the waterholes which creates in the end a json file with all my bounding boxes. Will be needed now to segment the image and the corresponding labels. 

Once the manual annotation is done, run the segmentation of the created png image with padding.

In [ ]:
import os
import yaml

import counting_boats.train

#from counting_boats.train import segment


#segment the png images
counting_boats.train.segment("config_train_Drive.yaml", train_val_split=0.8)
# counting_boats.train.segment("config_test_Drive.yaml", train_val_split=0.8)


After segmentation, we evaluate the results of the segmentation and production of material to train the model using the "train.describe" function. 
Run the bellow cell to describe from created paths of segmented images. 

In [ ]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.describe("config_train_Drive.yaml")
# counting_boats.train.describe("config_test_Drive.yaml")


Apply the cull command which will remove images with no labels until 10% of the training set has no labels. This has to be done post segmentation as we don't know prior the the amount (depends on the segmentation). 
Using my created function as the official cull from Charlie doesn't work well. Mine runs all good. 

In [ ]:
import sys
import os
import yaml
import random
import shutil
from pathlib import Path 

import counting_boats.train

#describe the created segmented images: 
# counting_boats.train.cull("config_train_GPU.yaml")
#AF: the cull function of Charlie seemed to run for ages... not sure why. So I develop an alternative one. 

counting_boats.train.cull_AF("config_train_Drive.yaml")


Once the cull function is applied reducing the no instance images amount to 10%, we can try to train the model. 
But first let's reorganise the folders to be used in the model. 

In [ ]:
import sys
import os
import yaml

import counting_boats.train

counting_boats.train.reorganize_folders("config_train_Drive.yaml")

Modifiy now in the config file of the yolo model training the path directory of the run. then run the following code and go in the yolov5 folder to run it and train the model. 

This code provides you with the command to excecute in the cmd of the yolo.  

In [ ]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.train("config_train_Drive.yaml")

Training of the model when ordered directly into the cmd panel and not via notebook: 
Need to figure out how to do it from here to streamline the whole process though...

In [ ]:
python train.py --workers 2 --img 416 --batch 8 --epochs 150 --data config_train_GPU_yolo.yaml --weights yolov5s.pt --cache disk

So it actually seems that it runs automatically with the function train but we do not see any progression... 
Prefer for now to run it manually in the cmd of the yolov5 folder.  

Changed the cache to the SSD drive we are using as we are limited in the storage available locally. 
Needed to set the project to the SSD which saves the outputs and doesn't increase the C: storage usage. 

Need to actually create the D:/temp and D:/yolo_run on your Drive or external directory 

In [ ]:
set TMPDIR=D:/temp
set TEMP=D:/temp
set TMP=D:/temp
set KMP_DUPLICATE_LIB_OK=TRUE 
python C:/Users/fossatia/Documents/Waterholes_project/yolov5/train.py --device cuda:0 --img 416 --batch 4 --workers 2 --epochs 250 --data C:\Users\fossatia\Documents\Waterholes_project\counting_waterholes\config_train_Drive.yaml --weights yolov5s.pt --cache False --project D:/yolo_runs

The set KMP_DUPLICATE_LIB_OK=TRUE is not recommended on the error command... I tried to google it and it seems we should force an install of the Nomkl using 'conda install nomkl --channel conda-forge'. 
However, by doing so, dependencies might be altered. To be checked. 

End of this script. 